In [2]:
import MaxText as mt
from MaxText import pyconfig
from MaxText import maxtext_utils
import numpy as np
from MaxText.input_pipeline import _input_pipeline_utils
import os
from MaxText.globals import PKG_DIR
from MaxText import max_logging
from MaxText import common_types
import jax

2025-04-25 18:18:21.628239: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745605101.647012 1122017 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745605101.652421 1122017 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745605101.667389 1122017 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745605101.667408 1122017 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745605101.667409 1122017 computation_placer.cc:177] computation placer alr

In [2]:
config = pyconfig.initialize(
    ["decode.py", "MaxText/configs/base.yml"], #TODO: Anisha: why decode.py?
    per_device_batch_size=1.0,
    run_name="test",
    enable_checkpointing=False,
    base_num_decoder_layers=2,
    max_target_length=16,
    base_emb_dim=256,
    base_num_query_heads=2,
    base_num_kv_heads=2,
    max_prefill_predict_length=4,
)

model, mesh, init_rng, *rest = mt.from_pretrained(config)
state, _ = maxtext_utils.setup_decode_state(model, config, init_rng, mesh, None)



Updating keys from env and command line: ['run_name', 'enable_checkpointing', 'base_emb_dim', 'base_num_query_heads', 'base_num_kv_heads', 'base_num_decoder_layers', 'per_device_batch_size', 'max_target_length', 'max_prefill_predict_length']
Running Model: default
Updating keys from model: []


Not using emergency checkpoint, ignoring local_checkpoint_directory, local_checkpoint_period, use_replicator_service and replicator_backup_interval_minutes
dataset_type set to tfds, will use keys['dataset_path']='' and keys['dataset_name']='c4/en:3.0.1'
Config param activations_in_float32: False
Config param adam_b1: 0.9
Config param adam_b2: 0.95
Config param adam_eps: 1e-08
Config param adam_eps_root: 0.0
Config param adam_weight_decay: 0.1
Config param add_bos: True
Config param add_eos: True
Config param allow_split_physical_axes: False
Config param ar_cache_axis_order: 1,2,0,3
Config param async_checkpointing: True
Config param attention: autoselected
Config param attention_type: global
Config param attn_logits_soft_cap: None
Config param autoregressive_decode_assert: 
Config param base_emb_dim: 256
Config param base_mlp_dim: 7168
Config param base_moe_mlp_dim: 7168
Config param base_num_decoder_layers: 2
Config param base_num_kv_heads: 2
Config param base_num_query_heads: 2
Confi

Config param inference_microbenchmark_loop_iters: 10
Config param inference_microbenchmark_num_samples: [1, 2, 3, 4, 5]
Config param inference_microbenchmark_prefill_lengths: 64,128,256,512,1024
Config param inference_microbenchmark_prefix_cache_common_prefix_proportion: 0.5
Config param inference_microbenchmark_prefix_cache_entries_num: 100
Config param inference_microbenchmark_stages: prefill,generate
Config param inference_server: MaxtextInterleavedServer
Config param init_weights_seed: 0
Config param jax_cache_dir: ~/jax_cache
Config param jax_debug_log_modules: 
Config param jax_distributed_initialization_timeout: 300
Config param jax_profiler_port: 9999
Config param key_proj: remat
Config param kv_lora_rank: 512
Config param kv_quant_axis: heads_and_dkv
Config param kv_quant_dtype: int8
Config param learning_rate: 3e-05
Config param learning_rate_schedule_steps: 150001
Config param load_balance_loss_weight: 0.01
Config param load_from_prefill_dir: False
Config param load_full_sta

In [3]:
source_tokenizer = _input_pipeline_utils.get_tokenizer(
        os.path.join(os.path.dirname(PKG_DIR), "assets", "tokenizer_llama3.tiktoken"),
        "tiktoken",
        add_bos=False,
        add_eos=False,
    )


#TODO: Anisha: any way to geto segment and position ids like HF tokenizer gives us?
input_ids = source_tokenizer.encode(config.prompt)#.numpy()
ids = np.asarray(input_ids, dtype=np.int32)
s = (config.global_batch_size_to_train_on, config.max_target_length)
decoder_segment_ids = np.zeros(s) + common_types.DECODING_ACTIVE_SEQUENCE_INDICATOR
decoder_positions = np.stack(
    [np.arange(config.max_target_length, dtype=np.int32) for _ in range(config.global_batch_size_to_train_on)]
)

#TODO: Anisha: simplify this config.global_batch_size_to_train_on=1
ids = np.stack([ids for _ in range(config.global_batch_size_to_train_on)])
max_logging.log(f"input_ids={input_ids}, ids={ids}, decoder_segment_ids = {decoder_segment_ids}, decoder_positions= {decoder_positions}")


Tokenizer path: /home/mazumdera/maxtext/assets/tokenizer_llama3.tiktoken
Reloaded tiktoken model from /home/mazumdera/maxtext/assets/tokenizer_llama3.tiktoken
#words: 128256 - BOS ID: 128000 - EOS ID: 128001
input_ids=[40, 3021, 311], ids=[[  40 3021  311]], decoder_segment_ids = [[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]], decoder_positions= [[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]]


In [1]:
import jax
!export TPU_LIBRARY_PATH=/home/mazumdera/.local/lib/python3.10/site-packages/libtpu/libtpu.so

jax.devices()

[CpuDevice(id=0)]

In [4]:
full_train_logits = model.apply(
          state.params,
          ids,
          decoder_positions,
          decoder_segment_ids,
          enable_dropout=False,
          rngs={"aqt": init_rng},
      )
full_train_logits = jax.experimental.multihost_utils.process_allgather(full_train_logits)
max_logging.log(f"{full_train_logits[0, 2, :]=}")

TypeError: mul got incompatible shapes for broadcasting: (1, 3, 2, 64), (1, 16, 1, 64).